# Perplexity OCS Bug Reproduction Tests

These cells intentionally reproduce known API errors to reproduce the GitHub Dimagi Open Chat Studio product integration error

In [ ]:
#          SONAR API Chat completions endpoint  --- SHOWING error 404

# Using requests library so it has error about wrong URL and 404 error (ie the chat/completions endpoint is not specified)
import requests
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("PERPLEXITY_API_KEY")

if not api_key:
    raise ValueError("PERPLEXITY_API_KEY environment variable not set")

url = "https://api.perplexity.ai/"  # this endpoint causes the 404 error
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
data = {
    "model": "sonar",  # Use a valid model from Perplexity docs
    "messages": [
        {"role": "user", "content": "What is day of week for 2024-06-01?"}
    ],
    "max_tokens": 50
}

response = requests.post(url, headers=headers, json=data)

if response.status_code == 200:
    result = response.json()
    print(f"Response ID: {result.get('id', 'N/A')}")
    print(result["choices"][0]["message"]["content"])
else:
    if response.status_code == 404:
        print("Error code 404: Endpoint not found. Please check the API documentation for the correct endpoint.")
    else:
        print(f"Error: {response.status_code} - {response.text}")

In [ ]:
#                 Perplexity Agent API with OpenAI SDK
#                         - with sonar model get error 400 (validation failed: model "sonar" is not supported')
# 
# See Sonar model NOT supported as using the Agent API responses endpoint https://api.perplexity.ai/v1
# For Sonar model url must be the chat completions endpoint (ie v1/chat/completions) 

from openai import OpenAI
import os
from dotenv import load_dotenv

# Load environment variables from .env file so dont need Linux variables.
load_dotenv()
perplex_api_key = os.getenv("PERPLEXITY_API_KEY")

client = OpenAI(
    api_key=perplex_api_key,
    base_url="https://api.perplexity.ai/v1"
)

try:
    response = client.responses.create(
        model="sonar",  # valid model for the Perplexity API
        input="In one sentence, what were the results of the 2025 French Open Finals?"
    )
    print(response.output_text)
except Exception as e:                       # catch HTTP/errors from the SDK
    print(f"API request failed ({type(e).__name__}): {e}")


In [ ]:
#                 Perplexity Agent API with OpenAI SDK chat completions endpoint
#                         - with sonar 
#                  - API request failed (NotFoundError): Error code: 404
# 
# For Perplexity Agent API , the end point must be https://api.perplexity.ai/v1 else it failes
# cant use the chat completions endpoint 

from openai import OpenAI
import os
from dotenv import load_dotenv

# Load environment variables from .env file so dont need Linux variables.
load_dotenv()
perplex_api_key = os.getenv("PERPLEXITY_API_KEY")

client = OpenAI(
    api_key=perplex_api_key,
    base_url="https://api.perplexity.ai/chat/completions"
)

try:
    response = client.responses.create(
        model="openai/gpt-5-mini",  # valid model for the Perplexity API
        input="In one sentence, what were the results of the 2025 French Open Finals?"
    )
    print(response.output_text)
except Exception as e:                       # catch HTTP/errors from the SDK
    print(f"API request failed ({type(e).__name__}): {e}")